# 2.3 — The 10-armed Testbed

---

## Why a testbed at all

A single bandit problem tells you almost nothing: any method can get lucky once. So we
generate **2000 independent bandit problems**, run each method on all of them for 1000
steps, and average. What we are estimating is *expected performance over the problem
distribution* — a method that happens to suit one draw of $q_*$ is averaged out.

The testbed:

$$q_*(a) \sim \mathcal{N}(0, 1), \quad a = 1,\dots,10 \qquad\qquad
R_t \mid A_t{=}a \;\sim\; \mathcal{N}\big(q_*(a),\, 1\big)$$

Two things we measure at each step $t$, averaged over runs:

1. **average reward** — the thing we actually care about;
2. **% optimal action** — did we pick $\arg\max_a q_*(a)$? This is the diagnostic; it
   tells you *why* the reward curve looks the way it does.

They can disagree, and when they do, it is informative.

## The asymptotic ceiling, again

For fixed $\varepsilon$, once $Q_t \to q_*$ the optimal-action rate settles at

$$\lim_{t\to\infty} P(A_t = a^*) \;=\; 1 - \varepsilon + \frac{\varepsilon}{k}$$

For $k=10$: $\varepsilon{=}0.1 \Rightarrow 91\%$, $\varepsilon{=}0.01 \Rightarrow 99.1\%$.
Keep those two numbers in mind — they are the whole story of Figure 2.2.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from bandit_utils import (Testbed, run_bandit, plot_pair, argmax_random_tiebreak,
                          hide, banner, RUNS, STEPS)

plt.rcParams["figure.dpi"] = 110
banner()

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not installed - run:  pip install ipywidgets")
    print("Static fallbacks are provided, so the notebook still works.")

In [ ]:
class EpsGreedy:
    def __init__(self, k, runs, rng, eps=0.1, Q_init=0.0, alpha=None):
        self.k, self.runs, self.rng, self.eps = k, runs, rng, eps
        self.alpha = alpha
        self.Q = np.full((runs, k), float(Q_init))
        self.N = np.zeros((runs, k))

    def act(self):
        greedy = argmax_random_tiebreak(self.Q, self.rng)
        rand = self.rng.integers(0, self.k, size=self.runs)
        explore = self.rng.random(self.runs) < self.eps
        return np.where(explore, rand, greedy)

    def update(self, a, r):
        idx = np.arange(self.runs)
        self.N[idx, a] += 1
        step = self.alpha if self.alpha else 1.0 / self.N[idx, a]
        self.Q[idx, a] += step * (r - self.Q[idx, a])

def eps_agent(eps, Q_init=0.0, alpha=None):
    return lambda k, runs, rng: EpsGreedy(k, runs, rng, eps=eps, Q_init=Q_init, alpha=alpha)

print("agent defined")

---

### Predict first

We are about to run $\varepsilon = 0$, $0.01$, and $0.1$ for 1000 steps.

1. Which one has the **highest average reward at step 100**?
2. Which one at **step 1000**?
3. Sketch, in your head, roughly where each of the three % optimal curves ends up.
4. Does $\varepsilon = 0$ plateau, or does it keep improving slowly?

*Write your guess down (mentally or in the cell below) before running the next cell. The point is not to be right — it is to make the surprise informative when you are wrong.*

In [ ]:
epsilons = [0.0, 0.01, 0.1]
results = [run_bandit(eps_agent(e), seed=s) for s, e in enumerate(epsilons)]
labels = [f"$\\varepsilon$ = {e}" for e in epsilons]

fig, axes = plot_pair(results, labels, title="Figure 2.2 - the 10-armed testbed")
for e, c in zip([0.01, 0.1], ["tab:orange", "tab:green"]):
    axes[1].axhline(100 * (1 - e + e / 10), ls=":", color=c, lw=1)
axes[1].text(520, 91.5, "$1-\\varepsilon+\\varepsilon/k$ ceilings", fontsize=8,
             color="gray")
plt.show()

for e, r in zip(epsilons, results):
    print(f"eps={e:<5} reward@100={r['rewards'][:100].mean():.3f} "
          f"reward@last100={r['rewards'][-100:].mean():.3f} "
          f"%opt@last100={r['optimal'][-100:].mean():.1f}")

In [ ]:
hide('''<b>Step 100:</b> the greedy method is still competitive &mdash; it spends every
step exploiting, and 100 steps is not long enough for its frozen estimates to cost it
much. &epsilon;=0.1 is paying 10% of its steps for information it has not cashed in yet.
<br><br><b>Step 1000:</b> the ordering has flipped. &epsilon;=0.1 wins on reward, and
&epsilon;=0.01 is climbing steadily and will eventually overtake it (its ceiling is 99.1%
vs 91%) &mdash; but it needs several thousand more steps to get there.
<br><br><b>&epsilon;=0:</b> it plateaus hard, around 1/3 optimal. It finds the best arm
about a third of the time and is permanently stuck otherwise. There is no mechanism by
which it improves.
<br><br>The lesson is not &quot;&epsilon;=0.1 is best&quot;. It is that <b>the best
&epsilon; depends on your horizon</b>. Short horizon &rarr; explore less. Long horizon
&rarr; explore more, or better, decay &epsilon;.''')

## The horizon dependence, made visible

If the best $\varepsilon$ depends on how long you get to play, we should be able to
*see* the crossover. Below: cumulative average reward, which is what you would actually
bank if you stopped at step $t$.

In [ ]:
eps_list = [0.0, 0.01, 0.05, 0.1, 0.4]
res = {e: run_bandit(eps_agent(e), steps=2000, seed=10 + i)
       for i, e in enumerate(eps_list)}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for e in eps_list:
    cum = np.cumsum(res[e]["rewards"]) / np.arange(1, 2001)
    axes[0].plot(cum, label=f"$\\varepsilon$={e}")
    axes[1].plot(res[e]["optimal"], label=f"$\\varepsilon$={e}", lw=1)
axes[0].set_xlabel("Horizon T (stop here)")
axes[0].set_ylabel("Cumulative average reward")
axes[0].set_title("Who wins depends on when you stop")
axes[0].set_xscale("log"); axes[0].set_xlim(10, 2000)
axes[1].set_xlabel("Steps"); axes[1].set_ylabel("% optimal action")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

best_at = {}
for T in [10, 50, 200, 1000, 2000]:
    scores = {e: res[e]["rewards"][:T].mean() for e in eps_list}
    best_at[T] = max(scores, key=scores.get)
print("best epsilon by horizon:", best_at)

That table is the single most useful thing in this section. **There is no universally
best $\varepsilon$** — only a best $\varepsilon$ for a given horizon and noise level.
Every subsequent method in this chapter is an attempt to get this right automatically
instead of by hand-tuning.

## Interactive: vary $\varepsilon$ and the reward noise

This is Exercise 2.3's territory. Turn the reward standard deviation down toward 0 and
watch what happens to the value of exploring.

In [ ]:
def testbed_experiment(eps=0.1, reward_sigma=1.0, steps=800, seed=0):
    res = run_bandit(eps_agent(eps), testbed_kwargs={"reward_sigma": reward_sigma},
                     steps=steps, seed=seed)
    base = run_bandit(eps_agent(0.0), testbed_kwargs={"reward_sigma": reward_sigma},
                      steps=steps, seed=seed)
    fig, axes = plot_pair([base, res], ["greedy ($\\varepsilon$=0)",
                                        f"$\\varepsilon$={eps}"],
                          title=f"reward noise $\\sigma$ = {reward_sigma}")
    axes[1].axhline(100 * (1 - eps + eps / 10), ls=":", color="gray")
    plt.show()

if HAVE_WIDGETS:
    W.interact(testbed_experiment,
               eps=W.FloatLogSlider(value=0.1, base=10, min=-3, max=-0.3, step=0.1,
                                    description="epsilon"),
               reward_sigma=W.FloatSlider(value=1.0, min=0.0, max=4.0, step=0.25,
                                          description="reward sd"),
               steps=W.fixed(800), seed=W.fixed(0))
else:
    for s in [0.0, 1.0, 3.0]:
        testbed_experiment(eps=0.1, reward_sigma=s)

### Thought experiment: noise-free rewards

*(Not Exercise 2.3 — that one asks you to quantify how much better $\varepsilon = 0.01$
eventually becomes than $\varepsilon = 0.1$, which the ceilings above answer: 99.1% versus
91% optimal actions.)*

> Suppose the reward variance were 0, so that you knew each $q_*(a)$ exactly after one
> pull. How would greedy and $\varepsilon$-greedy compare?

With $\sigma = 0$, one pull of each arm reveals the truth. But greedy never pulls each
arm — it pulls arm 1, gets $q_*(1)$ exactly, and if $q_*(1) > 0$ (its initial estimate
for everything else) it never moves again. So greedy still fails, but for a *different*
reason: not noisy estimates, but **never collecting the estimates at all**.

$\varepsilon$-greedy with $\sigma=0$ becomes near-optimal almost immediately — it takes
only $O(k \log k / \varepsilon)$ steps to have touched every arm once, and after that
its estimates are exact. Its remaining loss is pure exploration overhead, exactly
$\varepsilon(1 - 1/k) \cdot \mathbb{E}[\text{avg gap}]$ per step.

Run the slider at `reward sd = 0` to see both claims at once. This isolates a real
insight: **exploration serves two distinct purposes** — *coverage* (touch every arm at
least once) and *precision* (average out noise). They have different costs and
different remedies. Optimistic initial values (2.6) solve coverage very cheaply; UCB
(2.7) targets precision where it matters.

## Takeaways for 2.3

1. Averaging over 2000 problems is what makes any claim here meaningful.
2. Greedy plateaus around a third optimal; the plateau is structural, not slow learning.
3. The best $\varepsilon$ depends on the horizon — small $\varepsilon$ is slower but has
   a higher ceiling ($1-\varepsilon+\varepsilon/k$).
4. Exploration buys two different things: **coverage** and **precision**.

**Next:** the $\varepsilon$-greedy code above quietly hides an $O(1)$ trick for computing
running averages — Section 2.4 makes it explicit and turns it into the master update
rule of the entire book.